# Week 4 (starter): Multi-Tool Assistant

Everything below runs with no API key. The three tools, the validator, the dispatcher, and the dispatch loop are a **worked example** in a toy domain (arithmetic and unit conversion), driven by a scripted list of tool calls. They are the reference, not your submission.

Build your assistant in a domain you choose. The **TODO (you)** comments cover Parts 1 to 4; Part 5 is the submission checklist. The example uses only the Python standard library. For live calls through the OpenAI-compatible endpoint, install the client with `pip install openai`. Setting a key alone does not enable live calls.

In [1]:
import os, ast, operator, json, math
HAS_API_KEY = bool(os.environ.get('GEMINI_API_KEY', '').strip())
print('GEMINI_API_KEY set:', HAS_API_KEY)
# The scripted loop below runs either way. Wiring the live model call is yours (Part 1).

GEMINI_API_KEY set: True


In [32]:
def gemini_chat(messages, model='gemini-3.5-flash-lite', **kw):
    """Gemini via the OpenAI-compatible endpoint. Returns text, or None if no key (API-BLOCKED)."""
    key = os.environ.get('GEMINI_API_KEY')
    if not key:
        return None
    from openai import OpenAI
    client = OpenAI(api_key=key, base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    return client.chat.completions.create(model=model, messages=messages, **kw).choices[0].message

LIVE = os.environ.get('GEMINI_API_KEY') is not None
print('live model calls:', LIVE, '(fixtures used when False)')

live model calls: True (fixtures used when False)


## Part 1: Build the assistant
Define at least three tools with well-constrained JSON schemas, each using enums, required fields, and explicit types where they apply. Implement the request and response loop so the model can call a tool, receive the result, and continue toward an answer.

In [57]:
# Local function definitions with JSON schemas. Adapt these to your model API's tool format.
TOOLS = [
    {
        "name": "check_level_range",
        "description": "Determine whether a player is within the level range for an activity.",
        "parameters": {
            "type": "object",
            "properties": {
                "player_level": {
                    "type": "integer",
                    "minimum": 1,
                    "maximum": 60
                },
                "activity": {
                    "type": "string",
                    "enum": ["deadmines", "scarlet_monastery", "stratholme", "molten_core"]
                }
            },
            "required": ["player_level", "activity"],
            "additionalProperties": False
        }
    },
    {
        "name": "get_class_roles",
        "description": "Return the available group roles for a character class.",
        "parameters": {
            "type": "object",
            "properties": {
                "character_class": {
                    "type": "string",
                    "enum": [
                        "druid", "hunter", "mage", "paladin",
                        "priest", "rogue", "shaman", "warlock", "warrior"
                    ]
                }
            },
            "required": ["character_class"],
            "additionalProperties": False
        }
    },
    {
        "name": "calculate_xp_needed",
        "description": "Calculate XP needed using a arithmetic expression.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string"
                }
            },
            "required": ["expression"],
            "additionalProperties": False
        }
    }
]
print("tools:", [t["name"] for t in TOOLS])

tools: ['check_level_range', 'get_class_roles', 'calculate_xp_needed']


## Part 2: Add a guarded code-runner tool
Make one of the three tools execute code behind a guardrail: an allowlist of permitted operations, a check before execution, and a time limit. State plainly what your guardrail permits and what it blocks, naming the categories you block, for example filesystem, network, and process execution. This is a focused safety control, not a full production sandbox.

In [22]:
ACTIVITY_LEVELS = {
    "deadmines": (15, 25),
    "scarlet_monastery": (30, 45),
    "stratholme": (55, 60),
    "molten_core": (60, 60)
}

CLASS_ROLES = {
    "druid": ["tank", "healer", "damage"],
    "hunter": ["damage"],
    "mage": ["damage"],
    "paladin": ["tank", "healer", "damage"],
    "priest": ["healer", "damage"],
    "rogue": ["damage"],
    "shaman": ["healer", "damage"],
    "warlock": ["damage"],
    "warrior": ["tank", "damage"]
}

In [ ]:
import ast
from concurrent.futures import ThreadPoolExecutor, TimeoutError

def check_level_range(player_level, activity):
    min_level, max_level = ACTIVITY_LEVELS[activity]

    return {
        "eligible": min_level <= player_level <= max_level,
        "recommended_range": [min_level, max_level]
    }

def get_class_roles(character_class):
    return {
        "character_class": character_class,
        "roles": CLASS_ROLES[character_class]
    }

def calculate_xp_needed(expression):
    tree = ast.parse(expression, mode='eval')

    # Check the entire expression before execution
    validate_expression(tree)

    with ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(
            evaluate_expression,
            tree.body
        )

        try:
            xp_needed = future.result(timeout=1)
        except TimeoutError:
            raise TimeoutError("execution exceeded 1 second")

    return {
        "xp_needed": max(xp_needed, 0)
    }

_ALLOWED_NODES = (
    ast.Expression,
    ast.BinOp,
    ast.Sub,
    ast.Add,
    ast.Constant
)
def validate_expression(tree):
    for node in ast.walk(tree):
        if not isinstance(node, _ALLOWED_NODES):
            raise ValueError(
                f"operation not allowed: {type(node).__name__}"
            )
def evaluate_expression(node):
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node.op, ast.Add):
        return (
            evaluate_expression(node.left)
            + evaluate_expression(node.right)
        )

    if isinstance(node, ast.BinOp) and isinstance(node.op, ast.Sub):
        return evaluate_expression(node.left) - evaluate_expression(node.right)

    raise ValueError("only subtraction is allowed")


IMPL = {'check_level_range': check_level_range, 'get_class_roles': get_class_roles, "calculate_xp_needed": calculate_xp_needed}

class ToolArgError(Exception): pass
class ExecutionTimeout(Exception): pass

def timeout_handler(signum, frame):
    raise ExecutionTimeout("execution exceeded time limit")

# Validates the flat string/number schemas above. Extend this for your own schema types.
def validate(name, args):
    spec = next((t['parameters'] for t in TOOLS if t['name']==name), None)
    if spec is None: raise ToolArgError(f'unknown tool: {name}')
    if not isinstance(args, dict): raise ToolArgError('arguments must be a JSON object')
    for r in spec.get('required',[]):
        if r not in args: raise ToolArgError(f'missing required field: {r}')
    for k, v in args.items():
        p = spec['properties'].get(k)
        if p is None:
            raise ToolArgError(f'unexpected field: {k}')
        if p['type'] == 'string':
            if not isinstance(v, str):
                raise ToolArgError(f'{k} must be a string')
        elif p['type'] == 'integer':
            if type(v) is not int:
                raise ToolArgError(f'{k} must be an integer')
        else:
            raise ValueError(
                f'extend validate() to support schema type: {p["type"]}'
            )
        if 'enum' in p and v not in p['enum']:
            raise ToolArgError(
                f'{k}={v!r} not in {p["enum"]}'
            )
        
def dispatch(name, args):
    try:
        validate(name, args)
        output = IMPL[name](**args)
        # Make sure output can be returned to the model as JSON
        json.dumps(output, allow_nan=False)
        return {
            'ok': True,
            'tool': name,
            'output': output
        }
    except ToolArgError as e:
        return {
            'ok': False,
            'tool': name,
            'error_type': 'invalid_arguments',
            'message': str(e)
        }
    except Exception as e:
        return {
            'ok': False,
            'tool': name,
            'error_type': 'execution_error',
            'message': str(e)
        }

print(
    'happy path:',
    dispatch(
        'check_level_range',
        {
            'player_level': 32,
            'activity': 'scarlet_monastery'
        }
    )
)
print(
    'happy path:',
    dispatch(
        'get_class_roles',
        {'character_class': 'priest'}
    )
)
print(
    'guarded:',
    dispatch(
        'calculate_xp_needed',
        {'expression': 'open("secret.txt")'}
    )
)
print(
    'happy path:',
    dispatch(
        'calculate_xp_needed',
        {'expression': '18000 - 12000'}
    )
)
# TODO (you), Part 2: run_python shares the arithmetic interpreter above; it has no time limit.
# For your code-runner, check the whole input against an allowlist before evaluation and
# enforce a time limit before connecting model-generated inputs. Arithmetic can exhaust resources.
# State what your guard permits and what it blocks: filesystem, network, process execution.
#
# Guardrails:
# I implemented a guardrail for my calculate_xp_needed function. 
# This guardrail permits numeric constants and subtraction expressions only.
# It will block function calls, imports, names, attributes, and all other expressions
# not explicitly included in my AST allowlist.
#
# Allowed List:
# ast.Expression,
# ast.BinOp,
# ast.Sub,
# ast.Add,
# ast.Constant
#
# A timelimit is also enforced during evaluation to reduce resource exhaustion. 
# I used ThreadPoolExecutor package and set my timeout for 1 second. 

happy path: {'ok': True, 'tool': 'check_level_range', 'output': {'eligible': True, 'recommended_range': [30, 45]}}
happy path: {'ok': True, 'tool': 'get_class_roles', 'output': {'character_class': 'priest', 'roles': ['healer', 'damage']}}
guarded: {'ok': False, 'tool': 'calculate_xp_needed', 'error_type': 'execution_error', 'message': 'operation not allowed: Call'}
happy path: {'ok': True, 'tool': 'calculate_xp_needed', 'output': {'xp_needed': 6000}}


## Parts 1, 3, and 4: dispatch, evaluation, and recovery
The list below sends an invalid unit, then retries the same conversion with a valid unit. Both calls are scripted. No model chooses these calls or reads the errors, even when a key is set. Use this dispatch example to build your model loop, then evaluate your own tools and document a failure with recovery.

In [33]:
# TODO (you), Part 1: send the tools and query to the model, execute its tool calls, return
# the results with matching call IDs, and let the model continue until it gives an answer.
# TODO (you), Part 3: run at least three queries covering every tool. Include a query that
# needs two tools in sequence, where the second uses the first result. Show the call logs.
# TODO (you), Part 4: find a real failure in your own model's calls. Show the schema, bad
# call, cause, and successful recovery. Explain whether a schema change, description, or retry fixed it.

## Part 1 - Call Loop: 
send the tools and query to the model, execute its tool calls, return
the results with matching call IDs, and let the model continue until it gives an answer.

In [58]:
MODEL_TOOLS = [
    {
        "type": "function",
        "function": tool
    }
    for tool in TOOLS
]

def run_tool_loop(user_query):
    messages = [
        {
            "role": "user",
            "content": user_query
        }
    ]

    while True:
        message = gemini_chat(
            messages,
            tools=MODEL_TOOLS
        )

        if message is None:
            return "API key not found."

        if not message.tool_calls:
            return message.content

        messages.append(message)

        for tool_call in message.tool_calls:
            call_id = tool_call.id
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)

            print(
                f"CALL: {tool_name} "
                f"id={call_id} "
                f"args={tool_args}"
            )

            result = dispatch(tool_name, tool_args)

            print(
                f"RESULT: {tool_name} "
                f"id={call_id} "
                f"result={result}"
            )

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": call_id,
                    "content": json.dumps(result)
                }
            )

## Part 3: Evaluate
Run at least three queries that exercise each tool, including at least one that requires two tools in sequence. Show the tool-call log: which tool the model chose, the arguments it sent, and whether the call succeeded.

## Tool: check_level_range

In [47]:
print ("## Tool Example check_level_range ##")
answer1 = run_tool_loop(
    "I am level 32. Am I an appropriate level for Scarlet Monastery?"
)
print(answer1)



## Tool Example check_level_range ##
CALL: check_level_range id=call_1125360 args={'player_level': 32, 'activity': 'scarlet_monastery'}
RESULT: check_level_range id=call_1125360 result={'ok': True, 'tool': 'check_level_range', 'output': {'eligible': True, 'recommended_range': [30, 45]}}
Yes, you are! Level 32 is within the recommended level range for Scarlet Monastery, which is levels 30 to 45.


## Tool: get_class_roles

In [48]:
print ("## Tool Example get_class_roles ##")
answer2 = run_tool_loop(
    "What roles can a priest perform?"
)
print(answer2)

## Tool Example get_class_roles ##
CALL: get_class_roles id=call_1519427 args={'character_class': 'priest'}
RESULT: get_class_roles id=call_1519427 result={'ok': True, 'tool': 'get_class_roles', 'output': {'character_class': 'priest', 'roles': ['healer', 'damage']}}
A priest can perform the following roles:
- **Healer**
- **Damage**


## Tool: calculate_xp_needed

In [49]:
print ("## Tool Example calculate_xp_needed ##")
answer3 = run_tool_loop(
    "How much xp do I need to level. I have 3300 and need 6000"
)
print(answer3)

## Tool Example calculate_xp_needed ##
CALL: calculate_xp_needed id=call_1821283 args={'expression': '6000 - 3300'}
RESULT: calculate_xp_needed id=call_1821283 result={'ok': True, 'tool': 'calculate_xp_needed', 'output': {'xp_needed': 2700}}
You need 2,700 XP to level.


## Tool: Check_level_range and get_class_role

In [50]:
print ("## Tool Example calculate_xp_needed ##")
query_4 = (
    "I am a level 32 priest considering Scarlet Monastery. "
    "First check whether my level is appropriate for Scarlet Monastery. "
    "If it is appropriate, use my class to determine what roles I can perform."
)
answer4 = run_tool_loop(query_4)
print(answer4)

## Tool Example calculate_xp_needed ##
CALL: check_level_range id=call_1586002 args={'activity': 'scarlet_monastery', 'player_level': 32}
RESULT: check_level_range id=call_1586002 result={'ok': True, 'tool': 'check_level_range', 'output': {'eligible': True, 'recommended_range': [30, 45]}}
CALL: get_class_roles id=call_1966282 args={'character_class': 'priest'}
RESULT: get_class_roles id=call_1966282 result={'ok': True, 'tool': 'get_class_roles', 'output': {'character_class': 'priest', 'roles': ['healer', 'damage']}}
Your level (32) is appropriate for Scarlet Monastery (recommended range is 30 to 45). 

As a priest, the roles you can perform are **healer** and **damage**.


## Part 4: Find one failure and explain it
Document one real function-calling failure: an invalid argument that violates the schema, a hallucinated nested structure where a flat one was expected, or a runtime error in a tool. Show the schema, the bad call, why it failed, and the fix that let the model recover, whether that was a corrected schema, a clearer tool description, or a retry. Say which.

In [51]:
print ("## Tool Example calculate_xp_needed ##")
query_5 = (
    "Hi, i want to know how much experience I will have. I have 3000 right now and want to gain 6300 more."
)
answer5 = run_tool_loop(query_5)
print(answer5)

## Tool Example calculate_xp_needed ##
CALL: calculate_xp_needed id=call_1281811 args={'expression': '3000 + 6300'}
RESULT: calculate_xp_needed id=call_1281811 result={'ok': False, 'tool': 'calculate_xp_needed', 'error_type': 'execution_error', 'message': 'operation not allowed: Add'}
You will have a total of 9,300 experience!


**Schema:**
calculate_xp_needed accepts an expression string.
the tool description: "Calculate XP needed using a basic arithmetic expression."

**Bad call:**
the model generated "3000 + 6300", while the tool only allows for subtraction.

**Why it failed:**
although the string satisfied the JSON schema, the code-runner's AST allowlist permitted only ast.Sub, not ast.Add.

**What happened afterward:**
the tool returned an execution error to the model. The model continued and produced an answer without successfully using the tool.

**Fix:**
this is where I need to decide what behavior I want. Since the tool description says it calculates XP using a “basic arithmetic expression,” allowing only subtraction is inconsistent with that description.

I will update the tool description to say only subtractions. This should prevent the tool from being called and having to handle a tool failure.

In [54]:
print ("## Tool Example calculate_xp_needed ##")
query_5 = (
    "Hi, i want to know how much experience I will have. I have 3000 right now and want to gain 6300 more."
)
answer5 = run_tool_loop(query_5)
print(answer5)

## Tool Example calculate_xp_needed ##
CALL: calculate_xp_needed id=call_1120851 args={'expression': '3000 + 6300'}
RESULT: calculate_xp_needed id=call_1120851 result={'ok': False, 'tool': 'calculate_xp_needed', 'error_type': 'execution_error', 'message': 'operation not allowed: Add'}
CALL: calculate_xp_needed id=call_1954969 args={'expression': '9300 - 6300'}
RESULT: calculate_xp_needed id=call_1954969 result={'ok': True, 'tool': 'calculate_xp_needed', 'output': {'xp_needed': 3000}}
You will have **9,300** experience total (3,000 + 6,300).


The tooling loop is now calling the calculate_xp_needed. At this point, I am just going to allow for addition functionality, because this type of feature should be possible in an xp calculator. Also it would be better that the answer come from a determinstic source than the LLM do the math itself. 

In [59]:
print ("## Tool Example calculate_xp_needed ##")
query_5 = (
    "Hi, i want to know how much experience I will have. I have 3000 right now and want to gain 6300 more."
)
answer5 = run_tool_loop(query_5)
print(answer5)

## Tool Example calculate_xp_needed ##
CALL: calculate_xp_needed id=call_1450357 args={'expression': '3000 + 6300'}
RESULT: calculate_xp_needed id=call_1450357 result={'ok': True, 'tool': 'calculate_xp_needed', 'output': {'xp_needed': 9300}}
You will have a total of **9,300** experience.


## Part 5: Submit
Open a pull request with your schema design write-up, a link to your notebook, and a link to an issue documenting the failure and recovery. Describe your code-runner's allowlist, time limit, and blocked operations. Rubric: schemas (20), loop including a two-step sequence (25), guarded code-runner (20), failure with recovery (20), PR hygiene (15).